In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "50")

In [0]:
df = spark.read.table("`external-catalog`.default.employee_attrition")
display(df)

In [0]:
filtered_df = df.filter((df.Attrition == "No") & (df.JobSatisfaction.cast("int") < 3))
selected_df = filtered_df.select("EmployeeNumber", "JobRole", "JobSatisfaction", "Attrition", "Department", "Age", "MonthlyIncome")
selected_df.write.format("delta").mode("overwrite").saveAsTable("`external-catalog`.default.high_attrition_employees")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`external-catalog`.default.high_attrition_employees")
history_df = delta_table.history()
display(history_df)

In [0]:
df_delta = spark.read.format("delta").table("`external-catalog`.default.high_attrition_employees")
display(df_delta)

In [0]:
from pyspark.sql import Row

dummy_record = [Row(EmployeeNumber="999999", JobRole="DummyRole", JobSatisfaction="1", Attrition="No", Department="DummyDept", Age="30", MonthlyIncome="1000")]
dummy_df = spark.createDataFrame(dummy_record)
dummy_df.write.format("delta").mode("append").saveAsTable("`external-catalog`.default.high_attrition_employees")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`external-catalog`.default.high_attrition_employees")
versions_df = delta_table.history().select("version","timestamp","operation")
display(versions_df)

In [0]:
df_delta = spark.read.format("delta").table("`external-catalog`.default.high_attrition_employees")
display(df_delta)

In [0]:
df_version = spark.read.format("delta").option("versionAsOf", 0).table("`external-catalog`.default.high_attrition_employees")
display(df_version)

In [0]:
df_timestamp = spark.read.format("delta").option("timestampAsOf", "2026-04-19T10:54:51.000Z").table("`external-catalog`.default.high_attrition_employees")
display(df_timestamp)

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS `external-catalog`.default.employee_transformed_data")

In [0]:
from pyspark.sql.functions import col, when

# Logical transformation: Add a new column 'IsSenior' based on Age >= 40
transformed_df = df.withColumn("IsSenior", when(col("Age") >= 40, "Yes").otherwise("No"))

# Write to volume partitioned by Department
transformed_df.write.mode("overwrite").partitionBy("Department").parquet("/Volumes/external-catalog/default/employee_transformed_data/")